# Modern SLM Upgrades & Mixture of Experts
## Architecture-first take-home lab

This notebook follows one repeated learning loop:

**architecture picture → tensor shapes → equation → implementation → invariant test**

You will build and test:

- pre-norm residual wiring and RMSNorm,
- rotary position embeddings (RoPE),
- grouped-query attention (GQA),
- SwiGLU feed-forward networks,
- a readable top-$k$ Mixture-of-Experts layer,
- dense Llama-style and Mixtral-style decoder blocks.

**Runtime:** CPU is sufficient; a GPU makes the optional training comparison faster.  
**Deliverable:** an executed notebook, the final configuration dictionary, and the short architecture decision record at the end.

### How to work through the lab

At every **Checkpoint**, predict the outcome before running the cell. Do not skip assertions: each one encodes a mathematical or architectural invariant.

The notebook intentionally implements transparent teaching versions rather than fused production kernels. Shapes are written explicitly, and the MoE dispatcher uses a small Python loop over experts so that routing remains inspectable.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
import math
import random
from typing import Callable, Optional

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle, FancyArrowPatch
import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(7)
random.seed(7)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} · device={DEVICE}")

## 0. The decoder block is the unit we redesign

### 0A · Architecture picture

A decoder block performs two residual updates. Attention mixes information across token positions; the feed-forward branch transforms each token independently. Every upgrade in this lab occupies a precise site around this stable residual stream.

In [ ]:
def draw_decoder_block() -> None:
    fig = plt.figure(figsize=(12, 4.2))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, 12)
    ax.set_ylim(0, 4)
    ax.axis("off")

    def box(x: float, y: float, w: float, h: float, label: str) -> None:
        patch = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08")
        ax.add_patch(patch)
        ax.text(x + w / 2, y + h / 2, label, ha="center", va="center")

    ax.plot([0.6, 11.4], [2, 2], linewidth=2)
    ax.text(0.25, 2, "$x$", va="center", fontsize=13)
    ax.text(11.55, 2, "$x'$", va="center", fontsize=13)

    box(1.2, 3.0, 1.5, 0.6, "RMSNorm")
    box(3.1, 3.0, 1.9, 0.6, "Attention\nRoPE + GQA")
    ax.plot([0.9, 0.9, 1.2], [2, 3.3, 3.3])
    ax.plot([5.0, 5.8], [3.3, 2.0])
    ax.add_patch(Circle((5.8, 2.0), 0.22, fill=False))
    ax.text(5.8, 2.0, "+", ha="center", va="center")

    box(6.6, 0.45, 1.5, 0.6, "RMSNorm")
    box(8.5, 0.45, 2.0, 0.6, "SwiGLU\nor MoE")
    ax.plot([6.2, 6.2, 6.6], [2, 0.75, 0.75])
    ax.plot([10.5, 11.0], [0.75, 2.0])
    ax.add_patch(Circle((11.0, 2.0), 0.22, fill=False))
    ax.text(11.0, 2.0, "+", ha="center", va="center")

    ax.text(2.9, 3.78, "token mixing", ha="center")
    ax.text(8.6, 0.05, "token-wise computation", ha="center")
    plt.show()

draw_decoder_block()

### 0B · Tensor contract

Throughout the notebook, the residual stream has shape

$$x \in \mathbb{R}^{B\times T\times d}.$$

Each sublayer must return the same shape so that residual addition is defined. The next-token objective is not changed by any of the architectural upgrades.

# 1. Residual wiring and normalization

**What existed.** The 2017 transformer normalized *after* the residual addition, so a
normalization sat directly on the residual path.

**What broke.** With the norm on the path, the gradient crosses a normalization Jacobian at
every layer and the per-layer factors compound with depth — which is why post-norm models
needed a learning-rate warmup to train at all.

**What fixed it.** Move the norm onto the branch. The residual path then contributes an exact
identity term. GPT-2 already does this, so the wiring you built on Day 2 is pre-norm; the change
you make here is to *what the norm computes*, not where it sits.

## 1A · Architecture before algebra

**Post-norm** normalizes after the addition. **Pre-norm** normalizes only the learned branch input, leaving a direct identity route along the residual stream.

In [ ]:
def draw_norm_placement() -> None:
    fig = plt.figure(figsize=(12, 4))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, 12)
    ax.set_ylim(0, 4)
    ax.axis("off")

    def box(x, y, w, h, label):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08", fill=False))
        ax.text(x + w / 2, y + h / 2, label, ha="center", va="center")

    ax.text(2.7, 3.55, "post-norm", ha="center", fontsize=13)
    ax.plot([0.5, 5.4], [1.6, 1.6], linewidth=2)
    ax.plot([0.9, 0.9, 1.4], [1.6, 2.65, 2.65])
    box(1.4, 2.3, 1.6, 0.7, "$F$")
    ax.plot([3.0, 3.55], [2.65, 1.6])
    ax.add_patch(Circle((3.55, 1.6), 0.2, fill=False)); ax.text(3.55, 1.6, "+", ha="center", va="center")
    box(4.05, 1.25, 1.1, 0.7, "Norm")

    ax.text(9.1, 3.55, "pre-norm", ha="center", fontsize=13)
    ax.plot([6.6, 11.5], [1.6, 1.6], linewidth=2)
    ax.plot([7.0, 7.0, 7.5], [1.6, 2.65, 2.65])
    box(7.5, 2.3, 1.1, 0.7, "Norm")
    box(9.1, 2.3, 1.6, 0.7, "$F$")
    ax.plot([10.7, 11.1], [2.65, 1.6])
    ax.add_patch(Circle((11.1, 1.6), 0.2, fill=False)); ax.text(11.1, 1.6, "+", ha="center", va="center")
    ax.text(9.0, 0.75, "exact identity route", ha="center")
    plt.show()

draw_norm_placement()

## 1B · From wiring to equations

Post-norm:
$$x_{\ell+1}=\mathrm{Norm}\!\left(x_\ell+F_\ell(x_\ell)\right).$$

Pre-norm:
$$x_{\ell+1}=x_\ell+F_\ell\!\left(\mathrm{Norm}(x_\ell)\right).$$

Locally, the pre-norm residual derivative contains an explicit identity term:
$$\frac{\partial x_{\ell+1}}{\partial x_\ell}=I+J_{F_\ell}J_{\mathrm{Norm}}.$$

This is an optimization argument—not a universal quality guarantee.

## 1C · LayerNorm and RMSNorm

LayerNorm centers and rescales each token across its $d$ features:

$$\mathrm{LN}(x)_i=\gamma_i\frac{x_i-\mu}{\sqrt{\sigma^2+\varepsilon}}+\beta_i.$$

RMSNorm removes the centering operation:

$$\mathrm{RMSNorm}(x)_i=\gamma_i\frac{x_i}{\sqrt{\frac{1}{d}\sum_{j=1}^d x_j^2+\varepsilon}}.$$

**Toy prediction:** for $x=(1,2,3)$, LayerNorm creates negative/zero/positive values, while RMSNorm preserves the all-positive direction.

## 1D · Implementation exercise: RMSNorm

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6) -> None:
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: compute the mean square over the final feature dimension,
        # rescale x, and apply the learned weight.
        raise NotImplementedError("Implement RMSNorm.forward")

### Checkpoint 1 · Numerical geometry

With $\gamma=1$ and negligible $\varepsilon$:

- expected LayerNorm output: approximately $(-1.225,0,1.225)$,
- expected RMSNorm output: approximately $(0.463,0.926,1.389)$.

In [ ]:
toy = torch.tensor([[1.0, 2.0, 3.0]])
rms = RMSNorm(3, eps=1e-12)
with torch.no_grad():
    rms.weight.fill_(1.0)

rms_value = rms(toy)
ln_value = F.layer_norm(toy, normalized_shape=(3,), eps=1e-12)
print("LayerNorm:", ln_value.squeeze(0).tolist())
print("RMSNorm:  ", rms_value.squeeze(0).tolist())

expected_rms = torch.tensor([[0.462910, 0.925820, 1.388730]])
assert torch.allclose(rms_value, expected_rms, atol=1e-5)
assert torch.allclose(ln_value.mean(dim=-1), torch.zeros(1), atol=1e-6)
print("✓ RMSNorm toy values and LayerNorm centering verified")

In [ ]:
class PreNormResidual(nn.Module):
    def __init__(self, dim: int, sublayer: nn.Module) -> None:
        super().__init__()
        self.norm = RMSNorm(dim)
        self.sublayer = sublayer

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.sublayer(self.norm(x))

# With a zero branch, the local mapping is exactly y=x.
dim = 4
zero_branch = nn.Linear(dim, dim, bias=False)
nn.init.zeros_(zero_branch.weight)
wrapper = PreNormResidual(dim, zero_branch)
x = torch.randn(2, 3, dim, requires_grad=True)
y = wrapper(x)
y.sum().backward()
assert torch.allclose(y, x)
assert torch.allclose(x.grad, torch.ones_like(x))
print("✓ Zero branch exposes the exact identity path and identity gradient")

# 2. Rotary Position Embeddings (RoPE)

**What existed.** A position vector added to the token embedding once, before layer 0.

**What broke.** Expanding the attention logit shows content and position multiplying together in
cross terms; the logit depends on $m$ and $n$ separately rather than on $n-m$; and a learned
table has no rows beyond the trained context length.

**What fixed it.** Rotate queries and keys instead of adding to them. Rotations compose, so
$\langle R_m q, R_n k\rangle = q^{\top}R_{n-m}k$ — relative position falls out of absolute
rotations with no learned parameters and nothing added to the logit.

## 2A · Architecture picture

Absolute position embeddings add a position vector to the residual stream before Q/K/V projection. RoPE instead rotates pairs of query and key features **after projection**; values are not rotated.

In [ ]:
def draw_position_entry() -> None:
    fig = plt.figure(figsize=(12, 3.8))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, 12); ax.set_ylim(0, 3.8); ax.axis("off")

    def box(x, y, w, h, label):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08", fill=False))
        ax.text(x+w/2, y+h/2, label, ha="center", va="center")

    ax.text(2.8, 3.35, "absolute position", ha="center", fontsize=13)
    box(0.5, 2.0, 1.4, 0.65, "token")
    box(0.5, 0.75, 1.4, 0.65, "position")
    ax.text(2.4, 1.7, "+", fontsize=16)
    box(3.0, 1.4, 1.5, 0.65, "residual")
    box(5.0, 1.4, 1.0, 0.65, "Q K V")
    ax.annotate("", xy=(3.0,1.72), xytext=(1.9,2.32), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(3.0,1.72), xytext=(1.9,1.07), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(5.0,1.72), xytext=(4.5,1.72), arrowprops=dict(arrowstyle="->"))

    ax.text(9.2, 3.35, "rotary position", ha="center", fontsize=13)
    box(6.8, 1.4, 1.4, 0.65, "token")
    box(8.7, 1.4, 1.0, 0.65, "Q K V")
    box(10.3, 2.05, 1.2, 0.6, "rotate Q")
    box(10.3, 0.65, 1.2, 0.6, "rotate K")
    ax.annotate("", xy=(8.7,1.72), xytext=(8.2,1.72), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(10.3,2.35), xytext=(9.7,1.85), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(10.3,0.95), xytext=(9.7,1.55), arrowprops=dict(arrowstyle="->"))
    ax.text(10.85, 1.53, "V unchanged", ha="center")
    plt.show()

draw_position_entry()

## 2B · Rotation math

For one two-dimensional feature pair,

$$R(m\theta)=\begin{bmatrix}\cos(m\theta)&-\sin(m\theta)\\\sin(m\theta)&\cos(m\theta)\end{bmatrix}.$$

Because $R(m\theta)^\top R(n\theta)=R((n-m)\theta)$,

$$\langle R(m\theta)q,R(n\theta)k\rangle=q^\top R((n-m)\theta)k.$$

A shared position shift changes the absolute directions but not their relative angle.

## 2C · Implementation exercise: RoPE

> **Watch for:** RoPE is applied to queries and keys only. Rotating values is the single most
> common bug in this lab and it produces a model that trains without ever erroring.


In [ ]:
def rotate_half(x: torch.Tensor) -> torch.Tensor:
    '''Rotate adjacent feature pairs: (a,b) -> (-b,a).'''
    # TODO: split even and odd features, rotate, and interleave them again.
    raise NotImplementedError("Implement rotate_half")


def apply_rope_at_positions(
    x: torch.Tensor,
    positions: torch.Tensor,
    theta: float = 10_000.0,
) -> torch.Tensor:
    '''Apply RoPE to x shaped [B,H,T,D] at arbitrary integer positions [T].'''
    # TODO: construct inverse frequencies, angles, cos/sin, then rotate.
    raise NotImplementedError("Implement apply_rope_at_positions")

### Checkpoint 2 · Relative-angle intuition and a true RoPE invariant

First reproduce the two-dimensional lecture picture directly: rotate $q=k=(1,0)$ to $30^\circ$ and $90^\circ$, so the dot product is $\cos 60^\circ=0.5$. Then test the implementation with arbitrary RoPE frequencies: shifting **both** token positions by the same amount must leave the score unchanged.

In [ ]:
def rotate_2d(v: torch.Tensor, degrees: float) -> torch.Tensor:
    angle = math.radians(degrees)
    rotation = torch.tensor([
        [math.cos(angle), -math.sin(angle)],
        [math.sin(angle),  math.cos(angle)],
    ], dtype=v.dtype)
    return rotation @ v

unit = torch.tensor([1.0, 0.0])
toy_score = torch.dot(rotate_2d(unit, 30.0), rotate_2d(unit, 90.0))
print("2-D toy score:", toy_score.item())
assert torch.allclose(toy_score, torch.tensor(0.5), atol=1e-6)

torch.manual_seed(7)
q = torch.randn(1, 1, 1, 4)
k = torch.randn(1, 1, 1, 4)

def rope_score(m: int, n: int) -> torch.Tensor:
    q_m = apply_rope_at_positions(q, torch.tensor([m]))
    k_n = apply_rope_at_positions(k, torch.tensor([n]))
    return (q_m * k_n).sum(dim=-1)

score = rope_score(2, 7)
shifted = rope_score(9, 14)
print("score(2,7):", score.item())
print("score(9,14):", shifted.item())
assert torch.allclose(score, shifted, atol=1e-5)
print("✓ RoPE score depends on the relative offset, not a shared absolute shift")

# 3. Grouped-Query Attention (GQA)

**What existed.** One key head and one value head per query head.

**What broke.** Not quality — memory. During decoding every cached key and value must be read
back for each new token, so the KV cache dominates serving cost and decoding becomes limited by
memory bandwidth rather than arithmetic.

**What fixed it.** Let groups of query heads share a key/value pair. Multi-head and multi-query
are the two endpoints of that one dial, and the cache shrinks by exactly the head ratio.

## 3A · Architecture picture

Keep many query heads, but share fewer key/value heads. Multi-head attention (MHA) and multi-query attention (MQA) are the endpoints of the same design space.

In [ ]:
def draw_head_sharing() -> None:
    fig = plt.figure(figsize=(12, 4))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, 12); ax.set_ylim(0, 4); ax.axis("off")
    configs = [("MHA", 8), ("GQA", 2), ("MQA", 1)]
    for panel, (name, h_kv) in enumerate(configs):
        x0 = panel * 4 + 0.2
        ax.text(x0 + 1.7, 3.6, name, ha="center", fontsize=13)
        for q in range(8):
            x = x0 + 0.35 + (q % 4) * 0.9
            y = 2.65 - (q // 4) * 0.75
            ax.add_patch(Circle((x, y), 0.18, fill=False))
            ax.text(x, y, f"Q{q+1}", ha="center", va="center", fontsize=7)
        if h_kv == 8:
            for kv in range(8):
                x = x0 + 0.35 + (kv % 4) * 0.9
                y = 0.85 - (kv // 4) * 0.45
                ax.text(x, y, f"KV{kv+1}", ha="center", fontsize=8)
        elif h_kv == 2:
            ax.text(x0 + 0.9, 0.65, "KV1", ha="center")
            ax.text(x0 + 2.6, 0.65, "KV2", ha="center")
            ax.text(x0 + 1.75, 0.2, "four Q heads per KV", ha="center", fontsize=9)
        else:
            ax.text(x0 + 1.75, 0.65, "one shared KV", ha="center")
    plt.show()

draw_head_sharing()

## 3B · Tensor shapes and cache equation

$$Q\in\mathbb{R}^{B\times h_q\times T\times d_h},\qquad K,V\in\mathbb{R}^{B\times h_{kv}\times T\times d_h}.$$

The group size is $g=h_q/h_{kv}$. Each K/V head is reused by $g$ query heads.

During autoregressive decoding, the KV cache consumes

$$M_{KV}=2BLTh_{kv}d_hs\ \text{bytes},$$

where the factor $2$ counts keys and values and $s$ is bytes per scalar.

## 3C · Implementation exercise: GQA

> **Watch for:** expand shared key/value heads with `repeat_interleave` along the head axis, not
> `reshape`. A reshape silently interleaves the groups in the wrong order and the shapes still
> check out, so the causality and MHA-equivalence tests below are what will catch it.


In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(
        self,
        d_model: int,
        num_heads: int,
        num_kv_heads: int,
        rope_theta: float = 10_000.0,
    ) -> None:
        super().__init__()
        # TODO: validate divisibility, create Q/K/V/O projections, and store dimensions.
        raise NotImplementedError("Implement GroupedQueryAttention.__init__")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: project; reshape to [B,H,T,D]; apply RoPE to Q/K;
        # repeat K/V across query groups; run causal SDPA; merge heads.
        raise NotImplementedError("Implement GroupedQueryAttention.forward")

### Checkpoint 3 · MHA endpoint and cache arithmetic

When $h_{kv}=h_q$, GQA is ordinary MHA. The implementation should preserve shapes and causal masking in either case.

In [ ]:
def kv_cache_bytes(
    batch: int,
    layers: int,
    seq_len: int,
    num_kv_heads: int,
    head_dim: int,
    bytes_per_value: int,
) -> int:
    return 2 * batch * layers * seq_len * num_kv_heads * head_dim * bytes_per_value

mha_bytes = kv_cache_bytes(1, 24, 4096, 16, 64, 2)
gqa_bytes = kv_cache_bytes(1, 24, 4096, 4, 64, 2)
print(f"MHA: {mha_bytes / 2**20:.0f} MiB")
print(f"GQA: {gqa_bytes / 2**20:.0f} MiB")
assert mha_bytes == 4 * gqa_bytes

attention = GroupedQueryAttention(d_model=32, num_heads=4, num_kv_heads=4)
x = torch.randn(2, 7, 32)
y = attention(x)
assert y.shape == x.shape
assert torch.isfinite(y).all()
print("✓ MHA endpoint shape is valid; cache reduction is exactly 4× in the toy")

In [ ]:
# Causality test: changing future tokens must not affect earlier outputs.
attention.eval()
x_a = torch.randn(1, 8, 32)
x_b = x_a.clone()
x_b[:, 5:, :] = torch.randn_like(x_b[:, 5:, :])
with torch.no_grad():
    y_a = attention(x_a)
    y_b = attention(x_b)
assert torch.allclose(y_a[:, :5], y_b[:, :5], atol=1e-5, rtol=1e-5)
print("✓ Future-token perturbation cannot change earlier attention outputs")

# 4. SwiGLU: dense feature gating

**What existed.** Widen to $4d$, apply GELU elementwise, project back — roughly two thirds of
the block's parameters, with each feature deciding its own fate independently.

**What broke.** Nothing failed here. The question is whether a pointwise activation is the best
use of the largest parameter block in the model.

**What fixed it.** Gating: project twice in parallel, squash one branch, and multiply. Because
that costs a third matrix, the hidden width drops to $\tfrac{8}{3}d$ so the comparison stays at
matched parameters — which is the only kind of comparison that tells you anything.

## 4A · Architecture picture

A plain MLP has one hidden stream. SwiGLU creates a gate stream and a content stream, multiplies them elementwise, and projects back to $d$.

In [ ]:
def draw_swiglu() -> None:
    fig = plt.figure(figsize=(11, 3.3))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, 11); ax.set_ylim(0, 3.3); ax.axis("off")

    def box(x, y, w, h, label):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08", fill=False))
        ax.text(x+w/2, y+h/2, label, ha="center", va="center")

    box(0.4, 1.3, 0.8, 0.6, "$x$")
    box(2.0, 2.15, 1.7, 0.6, "$W_gx$ + SiLU")
    box(2.0, 0.45, 1.7, 0.6, "$W_ux$")
    ax.annotate("", xy=(2.0,2.45), xytext=(1.2,1.65), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(2.0,0.75), xytext=(1.2,1.55), arrowprops=dict(arrowstyle="->"))
    ax.text(4.6, 1.6, r"$\odot$", fontsize=20, ha="center")
    ax.annotate("", xy=(4.35,1.7), xytext=(3.7,2.45), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(4.35,1.5), xytext=(3.7,0.75), arrowprops=dict(arrowstyle="->"))
    box(5.25, 1.3, 1.3, 0.6, "$W_o$")
    box(7.3, 1.3, 0.8, 0.6, "$y$")
    ax.annotate("", xy=(5.25,1.6), xytext=(4.85,1.6), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(7.3,1.6), xytext=(6.55,1.6), arrowprops=dict(arrowstyle="->"))
    ax.text(9.4, 1.6, "dense gate over hidden features", ha="center")
    plt.show()

draw_swiglu()

## 4B · Equation and parameter matching

$$\mathrm{SwiGLU}(x)=W_o\!\left(\mathrm{SiLU}(W_gx)\odot W_ux\right).$$

Ignoring biases, a two-matrix MLP has about $2dd_{ff}$ parameters while SwiGLU has $3dd_{ff}$. Matching a $4d$ MLP gives

$$d_{ff}^{\text{SwiGLU}}\approx\frac{8d}{3}.$$

## 4C · Implementation exercise: SwiGLU

> **Watch for:** set the hidden width to $\tfrac{8}{3}d$, not $4d$. Leaving it at $4d$ inflates the
> parameter count by half and makes any comparison against the GELU MLP meaningless.


In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, d_model: int, d_ff: int) -> None:
        super().__init__()
        # TODO: create gate, content/up, and output projections without bias.
        raise NotImplementedError("Implement SwiGLU.__init__")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: apply SiLU to the gate stream, multiply by content, project down.
        raise NotImplementedError("Implement SwiGLU.forward")

In [ ]:
def count_parameters(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters())

d_model = 256
gpt_mlp = nn.Sequential(
    nn.Linear(d_model, 4 * d_model, bias=False),
    nn.GELU(),
    nn.Linear(4 * d_model, d_model, bias=False),
)
matched_width = round(8 * d_model / 3)
rounded_width = 704
swiglu = SwiGLU(d_model, rounded_width)

print("GPT MLP parameters: ", f"{count_parameters(gpt_mlp):,}")
print("8d/3 width:        ", matched_width)
print("SwiGLU parameters: ", f"{count_parameters(swiglu):,}")
assert swiglu(torch.randn(2, 5, d_model)).shape == (2, 5, d_model)
print("✓ SwiGLU preserves [B,T,d]")

# 5. Mixture of Experts: sparse token routing

**What existed.** One dense FFN, so every token pays for every parameter regardless of what
the token is.

**What broke.** Capacity and per-token compute are welded to the same axis: more parameters
always means more FLOPs per token.

**What fixed it.** Replace the FFN with $N$ experts and a router that activates $k \ll N$ of
them per token, so total parameters and active parameters come apart. The price is paid in
memory and in routing stability — every expert still has to be resident, and without a
load-balancing term the router collapses onto a few experts.

## 5A · Architecture picture

MoE replaces one dense FFN with a router plus several expert FFNs. For each token, the router selects only the top-$k$ experts and blends their outputs. The residual wrapper still sees $[B,T,d]\rightarrow[B,T,d]$.

In [ ]:
def draw_moe() -> None:
    fig = plt.figure(figsize=(12, 4.2))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, 12); ax.set_ylim(0, 4.2); ax.axis("off")

    def box(x, y, w, h, label):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08", fill=False))
        ax.text(x+w/2, y+h/2, label, ha="center", va="center")

    box(0.5, 1.75, 1.0, 0.7, "$x$")
    box(2.2, 1.75, 1.5, 0.7, "router")
    for i, (x, y) in enumerate([(5.0,3.0),(7.2,3.0),(5.0,0.6),(7.2,0.6)]):
        box(x, y, 1.4, 0.7, f"expert {i+1}")
    box(9.6, 1.75, 1.6, 0.7, "weighted sum")
    ax.annotate("", xy=(2.2,2.1), xytext=(1.5,2.1), arrowprops=dict(arrowstyle="->"))
    for x, y in [(5.0,3.35),(7.2,3.35),(5.0,0.95),(7.2,0.95)]:
        ax.annotate("", xy=(x,y), xytext=(3.7,2.1), arrowprops=dict(arrowstyle="->"))
        ax.annotate("", xy=(9.6,2.1), xytext=(x+1.4,y), arrowprops=dict(arrowstyle="->"))
    ax.text(6.9, 2.1, "top-k paths are active", ha="center")
    plt.show()

draw_moe()

## 5B · Routing equations

$$p(x)=\mathrm{softmax}(W_rx),\qquad S(x)=\mathrm{TopK}(p(x),k),$$

$$y=\sum_{i\in S(x)}\widetilde p_i(x)E_i(x).$$

For router health, track both the average probability $P_i$ and the realized assignment fraction $f_i$. A simplified Switch-style auxiliary term is

$$\mathcal L_{bal}=E\sum_{i=1}^{E}f_iP_i.$$

The code below uses assignment fractions over all top-$k$ slots, so they sum to one.

In [ ]:
@dataclass
class RouterStats:
    logits: torch.Tensor
    probabilities: torch.Tensor
    top_indices: torch.Tensor
    top_weights: torch.Tensor
    assignment_fraction: torch.Tensor
    mean_probability: torch.Tensor
    balance_loss: torch.Tensor

## 5C · Implementation exercise: Top-$k$ MoE

In [ ]:
class TopKMoE(nn.Module):
    def __init__(self, d_model: int, d_ff: int, num_experts: int, top_k: int) -> None:
        super().__init__()
        # TODO: validate top_k, create a bias-free router and expert SwiGLUs.
        raise NotImplementedError("Implement TopKMoE.__init__")

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, RouterStats]:
        # TODO:
        # 1. flatten tokens,
        # 2. compute router probabilities and top-k choices,
        # 3. renormalize selected weights,
        # 4. dispatch selected tokens to each expert,
        # 5. index-add weighted outputs,
        # 6. compute assignment fractions and balance loss.
        raise NotImplementedError("Implement TopKMoE.forward")

### Checkpoint 4 · One-token numerical routing toy

Router logits $[2.0,0.5,-1.0,1.0]$ produce probabilities approximately
$[0.609,0.136,0.030,0.224]$. The top two are experts 1 and 4, whose selected weights renormalize to approximately $[0.731,0.269]$.

In [ ]:
logits = torch.tensor([[2.0, 0.5, -1.0, 1.0]])
probabilities = logits.softmax(dim=-1)
chosen_prob, chosen_idx = probabilities.topk(2, dim=-1)
chosen_weight = chosen_prob / chosen_prob.sum(dim=-1, keepdim=True)
print("probabilities:", probabilities.squeeze(0).tolist())
print("top indices:  ", chosen_idx.squeeze(0).tolist())
print("top weights:  ", chosen_weight.squeeze(0).tolist())
assert chosen_idx.tolist() == [[0, 3]]
assert torch.allclose(chosen_weight, torch.tensor([[0.7310586, 0.2689414]]), atol=1e-6)
print("✓ Top-2 selection and renormalization match the lecture toy")

In [ ]:
moe = TopKMoE(d_model=16, d_ff=32, num_experts=4, top_k=2)
batch = torch.randn(3, 7, 16)
moe_output, stats = moe(batch)
print("output shape:", tuple(moe_output.shape))
print("assignment fraction:", stats.assignment_fraction.detach().tolist())
print("mean probability:    ", stats.mean_probability.detach().tolist())
print("balance loss:        ", float(stats.balance_loss.detach()))
assert moe_output.shape == batch.shape
assert torch.allclose(stats.assignment_fraction.sum(), torch.tensor(1.0), atol=1e-6)
assert torch.isfinite(moe_output).all()

plt.figure(figsize=(7, 3.5))
plt.bar(range(1, 5), stats.assignment_fraction.detach().cpu())
plt.xticks(range(1, 5))
plt.xlabel("expert")
plt.ylabel("assignment fraction")
plt.title("Router utilization for one random batch")
plt.show()

# 6. Assemble complete decoder blocks

## 6A · Architecture

A dense Llama-style block uses pre-RMSNorm, GQA with RoPE, and a dense SwiGLU FFN. A Mixtral-style block changes **only** the FFN branch to routed experts.

In [ ]:
class DenseLlamaBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int, d_ff: int) -> None:
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = GroupedQueryAttention(d_model, num_heads, num_kv_heads)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, d_ff)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.attn_norm(x))
        x = x + self.ffn(self.ffn_norm(x))
        return x


class MixtralStyleBlock(nn.Module):
    def __init__(
        self,
        d_model: int,
        num_heads: int,
        num_kv_heads: int,
        d_ff: int,
        num_experts: int,
        top_k: int,
    ) -> None:
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = GroupedQueryAttention(d_model, num_heads, num_kv_heads)
        self.ffn_norm = RMSNorm(d_model)
        self.moe = TopKMoE(d_model, d_ff, num_experts, top_k)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, RouterStats]:
        x = x + self.attn(self.attn_norm(x))
        moe_out, stats = self.moe(self.ffn_norm(x))
        x = x + moe_out
        return x, stats

## 6B · Five invariant tests

1. **Shape:** $[B,T,d]\rightarrow[B,T,d]$  
2. **Finite values:** no NaN or Inf  
3. **Causality:** future-token changes cannot affect earlier outputs  
4. **Gradients:** active paths receive finite gradients  
5. **Optimization smoke test:** a tiny next-token loss decreases over several steps

In [ ]:
def assert_block_invariants() -> None:
    dense = DenseLlamaBlock(d_model=32, num_heads=4, num_kv_heads=2, d_ff=80)
    sparse = MixtralStyleBlock(
        d_model=32, num_heads=4, num_kv_heads=2, d_ff=64, num_experts=4, top_k=2
    )
    x = torch.randn(2, 9, 32, requires_grad=True)

    dense_y = dense(x)
    sparse_y, stats = sparse(x)
    assert dense_y.shape == x.shape == sparse_y.shape
    assert torch.isfinite(dense_y).all() and torch.isfinite(sparse_y).all()

    # Causality.
    dense.eval()
    prefix = 5
    a = torch.randn(1, 9, 32)
    b = a.clone(); b[:, prefix:] = torch.randn_like(b[:, prefix:])
    with torch.no_grad():
        out_a = dense(a)
        out_b = dense(b)
    assert torch.allclose(out_a[:, :prefix], out_b[:, :prefix], atol=1e-5, rtol=1e-5)

    # Backward through dense and sparse paths.
    loss = dense_y.square().mean() + sparse_y.square().mean() + 0.01 * stats.balance_loss
    loss.backward()
    for name, param in list(dense.named_parameters()) + list(sparse.named_parameters()):
        if param.grad is not None:
            assert torch.isfinite(param.grad).all(), name
    assert sparse.moe.router.weight.grad is not None
    print("✓ shape · finite · causal · backward invariants passed")

assert_block_invariants()

# 7. Model-family fingerprints and the OLM bridge

The point is not to memorize every model report. Recognize a reusable core and a few deliberate deviations:

| Family | Attention | Normalization / position | FFN |
|---|---|---|---|
| Llama 3 | GQA | pre-RMSNorm, RoPE | dense SwiGLU |
| Qwen 3 | GQA with QK-Norm | pre-RMSNorm, RoPE | dense or MoE |
| Gemma 3 | local/global attention schedule with GQA | RMSNorm and QK-Norm | gated FFN |

These are **teaching fingerprints**, not claims of checkpoint compatibility.

## 7A · OLM composition

OpenLanguageModel expresses a Llama-style block through readable composition:

```python
from olm.nn.structure import Block
from olm.nn.structure.combinators import Residual
from olm.nn.attention import GroupedQueryAttention
from olm.nn.feedforward import SwiGLUFFN
from olm.nn.norms import RMSNorm

llama3_block = Block([
    Residual(Block([
        RMSNorm(embed_dim, eps=1e-5),
        GroupedQueryAttention(
            embed_dim, num_heads, num_kv_heads,
            max_seq_len, use_bias=False,
        ),
    ])),
    Residual(Block([
        RMSNorm(embed_dim, eps=1e-5),
        SwiGLUFFN(embed_dim, hidden_dim=intermediate_size, bias=False),
    ])),
])
```

OLM v2.2 documents named Llama 3.x presets. Qwen 3 and Gemma 3 should be presented here as component-level assemblies unless the installed OLM version adds named presets.

In [ ]:
try:
    import olm  # type: ignore
    print("OLM import succeeded:", getattr(olm, "__version__", "version not exposed"))
except ImportError:
    print("OLM is optional for this self-contained lab.")
    print("Install from the official repository before using the OLM composition cell.")

# 8. Tiny next-token training smoke test

This is not a benchmark. It checks that the complete block can participate in a language-model loop: embeddings → decoder block → final norm → LM head → cross-entropy.

In [ ]:
class TinyLanguageModel(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, block: nn.Module, is_moe: bool) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.block = block
        self.is_moe = is_moe
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, tokens: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        x = self.embedding(tokens)
        if self.is_moe:
            x, stats = self.block(x)
            aux = stats.balance_loss
        else:
            x = self.block(x)
            aux = torch.zeros((), device=x.device)
        logits = self.lm_head(self.final_norm(x))
        return logits, aux


def make_counting_batch(batch_size: int, seq_len: int, vocab_size: int, device: torch.device):
    start = torch.randint(0, vocab_size, (batch_size, 1), device=device)
    offsets = torch.arange(seq_len + 1, device=device)[None, :]
    sequence = (start + offsets) % vocab_size
    return sequence[:, :-1], sequence[:, 1:]


def train_smoke(model: TinyLanguageModel, steps: int = 16) -> list[float]:
    model.to(DEVICE)
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
    losses: list[float] = []
    for _ in range(steps):
        inputs, targets = make_counting_batch(12, 16, 32, DEVICE)
        logits, aux = model(inputs)
        lm_loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        loss = lm_loss + 0.01 * aux
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses.append(float(lm_loss.detach()))
    return losses


dense_model = TinyLanguageModel(
    vocab_size=32,
    d_model=32,
    block=DenseLlamaBlock(32, num_heads=4, num_kv_heads=2, d_ff=80),
    is_moe=False,
)
moe_model = TinyLanguageModel(
    vocab_size=32,
    d_model=32,
    block=MixtralStyleBlock(32, 4, 2, d_ff=64, num_experts=4, top_k=2),
    is_moe=True,
)

dense_losses = train_smoke(dense_model)
moe_losses = train_smoke(moe_model)
print(f"dense: {dense_losses[0]:.3f} → {dense_losses[-1]:.3f}")
print(f"MoE:   {moe_losses[0]:.3f} → {moe_losses[-1]:.3f}")
assert dense_losses[-1] < dense_losses[0]
assert moe_losses[-1] < moe_losses[0]

plt.figure(figsize=(8, 4))
plt.plot(dense_losses, label="dense")
plt.plot(moe_losses, label="MoE")
plt.xlabel("optimization step")
plt.ylabel("next-token cross-entropy")
plt.title("Tiny counting-task smoke test")
plt.legend()
plt.show()

# 9. Architecture decision record

Fill this in after running the notebook. Use numerical evidence from the cache calculator, parameter counts, assertions and training smoke test.

In [ ]:
architecture = {
    "block": "TODO: dense_llama_style or mixtral_style",
    "d_model": None,
    "num_layers": None,
    "num_query_heads": None,
    "num_kv_heads": None,
    "head_dim": None,
    "d_ff": None,
    "context_length": None,
    "normalization": "TODO",
    "position": "TODO",
    "ffn": "TODO",
    "moe": None,  # or {"num_experts": ..., "top_k": ..., "balance_weight": ...}
}
architecture

### Reflection prompts

1. **Architecture:** Draw your final block and label the two residual updates.  
2. **Memory:** Compute KV-cache bytes for batch size 1 at your context length and dtype.  
3. **Width:** Explain how you selected the SwiGLU intermediate size.  
4. **Risk:** Name the first failure mode you would inspect during training.  
5. **MoE decision:** If you selected MoE, justify the expert count, $k$, balance-loss weight and overflow policy. If you selected dense, state the evidence required before adding MoE.

# 10. Sources

Conceptual claims in this lab are grounded in established papers; model reports are used for architecture specifications.

- Xiong et al. (2020), **On Layer Normalization in the Transformer Architecture**, ICML. https://proceedings.mlr.press/v119/xiong20b.html
- Zhang & Sennrich (2019), **Root Mean Square Layer Normalization**, NeurIPS. https://proceedings.neurips.cc/paper/2019/hash/1e8a19426224ca89e83cef47f1e7f53b-Abstract.html
- Su et al. (2024), **RoFormer: Enhanced Transformer with Rotary Position Embedding**, Neurocomputing. https://doi.org/10.1016/j.neucom.2023.127063
- Ainslie et al. (2023), **GQA: Training Generalized Multi-Query Transformer Models**, EMNLP. https://aclanthology.org/2023.emnlp-main.298/
- Dauphin et al. (2017), **Language Modeling with Gated Convolutional Networks**, ICML. https://proceedings.mlr.press/v70/dauphin17a.html
- Shazeer et al. (2017), **Outrageously Large Neural Networks: The Sparsely-Gated Mixture-of-Experts Layer**, ICLR. https://openreview.net/forum?id=B1ckMDqlg
- Fedus et al. (2022), **Switch Transformers**, JMLR. https://jmlr.org/papers/v23/21-0998.html
- Jiang et al. (2024), **Mixtral of Experts**. https://arxiv.org/abs/2401.04088
- Llama 3 report. https://arxiv.org/abs/2407.21783
- Qwen 3 report. https://arxiv.org/abs/2505.09388
- Gemma 3 report. https://arxiv.org/abs/2503.19786
- OpenLanguageModel documentation. https://openlanguagemodel.github.io/openlanguagemodel/

## Submission checklist

- [ ] All required TODOs are complete.
- [ ] Every checkpoint assertion passes.
- [ ] The router-utilization and training-loss plots are visible.
- [ ] The architecture dictionary is filled.
- [ ] The five reflection prompts are answered.